In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.select import Select
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

import winsound as sd
def beepsound():
    fr = 2000
    du = 1000
    sd.Beep(fr, du)

In [ ]:
srt_home = "https://etk.srail.kr/main.do"
srt_login = "https://etk.srail.kr/cmc/01/selectLoginForm.do?pageId=TK0701000000"
srt_reservation = "https://etk.srail.kr/hpg/hra/01/selectConditionGoodsScheduleList.do?pageId=TK0101060000"
id = # id(숫자)
pw = # 비밀번호(문자열)

depart_station = '동대구' # 출발역
arrive_station = '수서' # 도착역
depart_date = '20230414' # 날짜
depart_time_1 = '1800' # 출발시간1
depart_time_2 = '2000' # 출발시간2

In [ ]:
# 드라이버 위치 경로 입력

def login(driver, id, pw):
    driver.get(srt_login)
    driver.implicitly_wait(3)
    
    driver.find_element('xpath', '//*[@id="srchDvNm01"]').send_keys(id)
    driver.find_element('xpath', '//*[@id="hmpgPwdCphd01"]').send_keys(pw)
    driver.find_element('xpath', '//*[@id="login-form"]/fieldset/div[1]/div[1]/div[2]/div/div[2]/input').click()
    
    driver.implicitly_wait(3)

def select_date(driver, depart_station, arrive_station, depart_date, depart_time_1):
    driver.get(srt_reservation)
    
    person_nbr = Select(driver.find_element('xpath', '//*[@id="pbs03"]/div[2]/div[1]/select'))
    person_nbr.select_by_value('1')
    
    driver.find_element('xpath', '//*[@id="dptRsStnCdNm"]').clear()
    driver.find_element('xpath', '//*[@id="dptRsStnCdNm"]').send_keys(depart_station)
    driver.find_element('xpath', '//*[@id="arvRsStnCdNm"]').clear()
    driver.find_element('xpath', '//*[@id="arvRsStnCdNm"]').send_keys(arrive_station)
    
    driver.implicitly_wait(3)
    
    select_date = Select(driver.find_element('xpath', '//*[@id="dptDt"]'))
    select_date.select_by_value(depart_date)
    
    
    select_time = Select(driver.find_element('xpath', '//*[@id="dptTm"]'))
    select_time.select_by_value(str(int(depart_time_1[:2])//2*20000))
    
    driver.find_element(By.XPATH, '//*[@id="search_top_tag"]/input').click()
    driver.implicitly_wait(3)

def get_next_page(pgnbr, driver):
    print('Get the Page = ',pgnbr+1, end=' ')
    if pgnbr == 1:
        nextButton = driver.find_element('xpath', '//*[@id="result-form"]/fieldset/div[8]/input')
    else: 
        nextButton = driver.find_element('xpath', '//*[@id="result-form"]/fieldset/div[8]/input[2]')
    print(nextButton.is_enabled())
    nextButton.click()
    driver.implicitly_wait(3)
    print('---done')
    return driver

def get_time_list(driver, depart_time1, depart_time2):
    print('Get Time List', end=' ')
    idxlist = []
    i = 1
    try:
        while(True):
            time = driver.find_element('xpath', f'//*[@id="result-form"]/fieldset/div[6]/table/tbody/tr[{i}]/td[4]/em')
            time = time.text.replace(':','')
            print(time, end=' ')
            if (int(time) >= int(depart_time1)) and (int(time) <= int(depart_time2)):
                idxlist.append(i)
            i += 1
    except:
        driver.implicitly_wait(1)
        print()
    return idxlist

def click_reservationButton(driver, idx):
    print(idx)
    while True:
        try:
            reservationButton = driver.find_element('xpath', f'//*[@id="result-form"]/fieldset/div[6]/table/tbody/tr[{idx}]/td[7]/a') # 예약하기 누르기
            reservationButton.click()
            driver.refresh()
            driver.implicitly_wait(1)
        except:
            return      
    
def make_reservation(depart_time1, depart_time2):
    pgnbr = 1
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    print('Settings...', end=' ')
    login(driver, id, pw)
    print('Login success')
    select_date(driver, depart_station, arrive_station, depart_date, depart_time_1)
    print('Date setting success')
    print('done...', end='\n\n')
    while True:
        times = get_time_list(driver, depart_time1, depart_time2)
        print('times = ', times)
                
        
        if len(times) == 0:
            driver = get_next_page(pgnbr, driver)
            pgnbr += 1
        if len(times) > 0:
            for idx in times:
                print('LEGO!!!!!!!!!!!!!!!!!!!!')
                click_reservationButton(driver, idx)
                beepsound()
            


In [ ]:
make_reservation(depart_time_1, depart_time_2)